# Pipeline for data ingestion of the live portfolio

Steps:
- Load data and split them
- Apply fixes to close price series based on params.py
- Translate price data into returns
- Apply eligibility criteria of at least 2 years of daily returns
- Regularize data to last trading day of each month
- Convert risk free data to same format as ETFs
- Estimate trading costs

In [120]:
# Imports

import pandas as pd
import numpy as np
import yfinance as yf
from vportfolio.params import UNIVERSE, BENCHMARK, RISK_FREE, DATA_FIXES, LOOKBACK, RESEARCH_START, RESEARCH_END, COST_OVERRIDES

In [121]:
# split raw data into partitions needed to run the engine
# returns a dictionary with the raw data
def split_prices(raw):
    # split the data and saves them through .copy so you can fix one without changing raw_data
    return {"universe_close" : raw["Close"][UNIVERSE].copy(),
            "universe_open" : raw["Open"][UNIVERSE].copy(),
            "benchmark_close" : raw["Close"][BENCHMARK + ["SPY"]].copy(),
            "irx" : raw["Close"][RISK_FREE[0]].copy()}

In [122]:
# load the raw data for comparison
raw_data = pd.read_parquet("../data/raw/yahoo_prices_2026-10-23.parquet")

In [123]:
#  Cross check with current processed file
prices = split_prices(raw_data)
prices["universe_open"].equals(pd.read_parquet(PROCESSED / "open.parquet"))

True

In [124]:
# function testing 

# fake raw data shaped like the Yahoo download: ("Close", ticker) and ("Open", ticker) columns
days = pd.bdate_range("2024-01-01", periods=3)
all_tickers = UNIVERSE + BENCHMARK + RISK_FREE
rng = np.random.default_rng(0)
close_fake = pd.DataFrame(rng.uniform(10, 100, (3, len(all_tickers))), index=days, columns=all_tickers)
open_fake = pd.DataFrame(rng.uniform(10, 100, (3, len(all_tickers))), index=days, columns=all_tickers)
raw_fake = pd.concat({"Close": close_fake, "Open": open_fake}, axis=1)

parts = split_prices(raw_fake)

# 1. always true: the four parts, each on the same dates as the raw data
assert set(parts.keys()) == {"universe_close", "universe_open", "benchmark_close", "irx"}
assert all(p.index.equals(days) for p in parts.values())

# 2. known answer: the right tickers and the same values as the raw data
assert list(parts["universe_close"].columns) == UNIVERSE
assert list(parts["universe_open"].columns) == UNIVERSE
assert list(parts["benchmark_close"].columns) == BENCHMARK + ["SPY"]
assert parts["universe_close"].equals(close_fake[UNIVERSE])
assert parts["universe_open"].equals(open_fake[UNIVERSE])
assert parts["irx"].equals(close_fake[RISK_FREE[0]])

# 3. inputs matter: the parts are copies, so changing one leaves the raw data untouched
parts["universe_close"].iloc[0, 0] = -1.0
assert raw_fake["Close"][UNIVERSE].iloc[0, 0] != -1.0

print("split_prices passed all tests")

split_prices passed all tests


In [125]:
# apply the manual data fixes listed in params.DATA_FIXES
# returns a copy of close with the fixes applied
def apply_fixes(close, fixes=DATA_FIXES):
    close = close.copy()
    for ticker, bad_date, copy_from in fixes:
        if pd.Timestamp(bad_date) in close.index and pd.Timestamp(copy_from) in close.index:
            close.loc[bad_date, ticker] = close.loc[copy_from, ticker]
    return close

In [126]:
#  Cross check with current processed file
close = apply_fixes(prices["universe_close"])
close.equals(pd.read_parquet(PROCESSED / "close.parquet"))

True

In [127]:
# testing function
days = pd.bdate_range("2007-06-18", periods=4) # 18, 19, 20, 21 June 2007
close = pd.DataFrame({"UUP": [25.0, 25.1, 99.0, 25.2], # 20 June: the bad price
                      "SPY": [150.0, 151.0, 152.0, 153.0]}, index=days)
original = close.copy()
fixed = apply_fixes(close)
# 1. always true: same shape, and the input is not modified
assert fixed.shape == close.shape and close.equals(original)
# 2. known answer: the bad UUP price takes the previous day's close
assert fixed.loc["2007-06-20", "UUP"] == 25.1
# 3. inputs matter: only that one cell changes; data without that date passes through unchanged
assert (fixed != close).sum().sum() == 1
recent = close.copy()
recent.index = pd.bdate_range("2026-01-05", periods=4)
assert apply_fixes(recent).equals(recent)
print("apply_fixes passed all tests")

apply_fixes passed all tests


In [128]:
# translate price data into daily returns
# returns a dataframe with returns calculated from closing prices
def make_returns(close):
    return close.pct_change(fill_method=None)

In [129]:
# call the function twice on universe and benchmark data
returns = make_returns(close)
benchmark_returns = make_returns(prices["benchmark_close"])

In [130]:
# cross check with the current processed files
print(returns.equals(pd.read_parquet(PROCESSED / "returns.parquet")))
print(benchmark_returns.equals(pd.read_parquet(PROCESSED / "benchmark_returns.parquet")))

False
True


In [131]:
# function testing
days = pd.bdate_range("2024-01-01", periods=4)
close = pd.DataFrame({"A": [100.0, 110.0, 99.0, 99.0],
                      "B": [None, None, 50.0, 55.0]}, index=days)  # B starts trading later
returns = make_returns(close)
# 1. always true: same dates and tickers as the prices; the first day has no return
assert returns.index.equals(close.index) and returns.columns.equals(close.columns)
assert returns.iloc[0].isna().all()
# 2. known answer: 100 → 110 is +10%, 110 → 99 is −10%, unchanged is 0
assert np.allclose(returns["A"].iloc[1:], [0.10, -0.10, 0.0])
# 3. inputs matter: no return before an ETF exists (no filling); its first return is on its second day
assert returns["B"].iloc[:3].isna().all()
assert np.isclose(returns["B"].iloc[3], 0.10)
print("make_returns passed all tests")

make_returns passed all tests


In [132]:
# apply the eligibility criteria of at least 2 years of daily returns
# returns a dataframe with eligible returns
def make_eligible(returns, lookback=LOOKBACK):
    return returns.notna().cumsum() >= lookback

In [133]:
# cross check with the current processed files
eligible = make_eligible(returns)
eligible.equals(pd.read_parquet(PROCESSED / "eligible.parquet"))

False

In [134]:
# function testing
days = pd.bdate_range("2024-01-01", periods=6)
returns = pd.DataFrame({"A": [None, 0.01, 0.01, 0.01, 0.01, 0.01],
                        "B": [None, None, None, None, 0.01, 0.01]}, index=days)  # B starts later
eligible = make_eligible(returns, lookback=3)
# 1. always true: True/False only; once eligible, an ETF stays eligible
assert eligible.dtypes.eq(bool).all()
assert (eligible.astype(int).diff().fillna(0) >= 0).all().all()
# 2. known answer: A reaches 3 returns on day 4; B only has 2, so it is never eligible
assert list(eligible["A"]) == [False, False, False, True, True, True]
assert not eligible["B"].any()
# 3. inputs matter: a shorter lookback makes A eligible a day sooner
assert make_eligible(returns, lookback=2)["A"].iloc[2] and not eligible["A"].iloc[2]
print("make_eligible passed all tests")

make_eligible passed all tests


In [135]:
# Check last trading date of each month and regularize
# returns a Series of rebalance dates (plain 0, 1, 2... index, like rebal_dates.parquet)
def make_rebal_dates(index, start, end=None):
    dates = index[index >= pd.Timestamp(start)]
    if end is not None:
        dates = dates[dates <= pd.Timestamp(end)]
    monthly = dates.to_series().groupby([dates.year, dates.month]).max().reset_index(drop=True)
    if monthly.empty:
        raise ValueError(f"no trading days between {start} and {end}")
    # drop the last month if the data stops before its last business day (incomplete month)
    last = monthly.iloc[-1]
    if last < last + pd.offsets.BMonthEnd(0):
        monthly = monthly.iloc[:-1]
    return monthly

In [136]:
# cross check with the current processed files
returns = pd.read_parquet(PROCESSED / "returns.parquet")
print(returns.index[0].date(), returns.index[-1].date(), len(returns)) # expect 2003 … 2026, thousands of rows

rebuilt = make_rebal_dates(returns.index, RESEARCH_START, RESEARCH_END)
saved = pd.read_parquet(PROCESSED / "rebal_dates.parquet")["date"]
assert len(rebuilt) == len(saved) == 204
assert (rebuilt.values == saved.values.astype(rebuilt.values.dtype)).all()
print("rebuilt dates match the processed file")

2003-01-02 2026-10-22 5990
rebuilt dates match the processed file


In [137]:
# function testing
days = pd.bdate_range("2024-01-01", "2024-03-15") # data stops mid-March
dates = make_rebal_dates(days, "2024-01-01")
# 1. always true: sorted, and every date is a real day in the data
assert dates.is_monotonic_increasing and dates.isin(days).all()
# 2. known answer: January and February month-ends; March incomplete, so dropped
assert list(dates) == [pd.Timestamp("2024-01-31"), pd.Timestamp("2024-02-29")]
# 3. inputs matter: data reaching March's last business day keeps March
full = pd.bdate_range("2024-01-01", "2024-03-29")
assert make_rebal_dates(full, "2024-01-01").iloc[-1] == pd.Timestamp("2024-03-29")
print("make_rebal_dates passed all tests")

make_rebal_dates passed all tests


In [138]:
# Convert the 13-week T-bill yield into a daily risk-free return on the ETF trading days
# returns a Series named "rf"
def make_rf_daily(irx, index):
    rf = (irx.ffill() # fill bond-market holidays with the previous yield
          / 100 # from % to decimal
          / 252) # annual to daily
    return rf.reindex(index).ffill().rename("rf") # align with the ETF trading days, fill any gaps

In [139]:
# cross check with the current processed files
irx = raw_data["Close"]["^IRX"] # the yield series from the raw download
rf_daily = make_rf_daily(irx, returns.index) # every trading day, not the rebalance dates

saved_rf = pd.read_parquet(PROCESSED / "rf_daily.parquet")["rf"]
assert np.allclose(rf_daily.loc[saved_rf.index], saved_rf, equal_nan=True)
print("rf_daily matches the processed file")

rf_daily matches the processed file


In [140]:
# function testing
days = pd.bdate_range("2024-01-01", periods=5)
irx = pd.Series([5.0, None, 5.0, 4.0, 4.0], index=days) # a missing day, then a lower yield
rf = make_rf_daily(irx, days)
# 1. always true: no gaps on trading days
assert rf.notna().all()
# 2. known answer: 5% a year → 0.05 / 252 a day; the missing day takes the previous value
assert np.isclose(rf.iloc[0], 0.05 / 252) and np.isclose(rf.iloc[1], 0.05 / 252)
# 3. inputs matter: a lower yield gives a lower rate
assert rf.iloc[3] < rf.iloc[0]
print("make_rf_daily passed all tests")

make_rf_daily passed all tests


In [141]:
# estimate trading costs as per pre-registration
# returns a Series of per-ETF costs in bp, fixed from research-period dollar volume
def estimate_costs(raw_data, tickers, overrides = COST_OVERRIDES):
    volume = (raw_data["Close"][tickers] * raw_data["Volume"][tickers]).loc[RESEARCH_START:RESEARCH_END].median()
    cost_bp = pd.cut(volume, bins=[0, 100e6, 1e9, float("inf")], labels=[10, 5, 2]).astype(int)
    cost_bp[overrides] = 2
    return cost_bp

In [142]:
cost_bp = estimate_costs(raw_data, UNIVERSE + BENCHMARK)
saved_cost = pd.read_parquet(PROCESSED / "cost_bp.parquet")["cost_bp"]
assert cost_bp.loc[saved_cost.index].equals(saved_cost.astype(cost_bp.dtype))
print("cost_bp matches the processed file")

cost_bp matches the processed file


In [143]:
# function testing
# fake raw data shaped like the Yahoo download: ("Close", ticker) and ("Volume", ticker) columns
days = pd.bdate_range("2005-01-03", "2023-12-29")
close = pd.DataFrame(1.0, index=days, columns=["A", "B", "C", "SHY"])   # price 1 → dollar volume = volume
volume = pd.DataFrame({"A": 50e6, "B": 500e6, "C": 2e9, "SHY": 50e6}, index=days)
raw_fake = pd.concat({"Close": close, "Volume": volume}, axis=1)
tickers = ["A", "B", "C", "SHY"]

cost = estimate_costs(raw_fake, tickers, overrides=["SHY"])
# 1. always true: only the three pre-registered levels
assert set(cost.unique()) <= {2, 5, 10}
# 2. known answers: $50m → 10 bp, $500m → 5 bp, $2bn → 2 bp; SHY overridden to 2 bp despite low volume
assert cost["A"] == 10 and cost["B"] == 5 and cost["C"] == 2 and cost["SHY"] == 2
# 3. inputs matter: volume after the research period is ignored
raw_later = raw_fake.copy()
raw_later.loc["2022-01-01":, ("Volume", "A")] = 1e12              # A becomes hugely liquid after 2021
assert estimate_costs(raw_later, tickers, overrides=["SHY"])["A"] == 10
print("estimate_costs passed all tests")

estimate_costs passed all tests
